# Large qubit chip demo: eigenmodes with saved fields

Simulate a **5.4 × 3.1 mm QPDK chip** with two qubits, readout resonators,
a shared bus, and vortex-trapping fill. Find four eigenmodes using
**2 µm refinement** near the metal and save the fields for all four modes.
The [two-qubit layout](https://github.com/gdsfactory/gsim/blob/main/nbs/data/qpdk/two_qubit_straight_bus.pic.yml)
is a straight-bus variant of QPDK's filled test chip.

**Requirements:**

- QPDK: `uv pip install qpdk`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

The model uses planar PEC metal on silicon with vacuum above and reflecting
PMC outer boundaries. Junctions are omitted, and both launcher ports have
matched 50 Ω loads, so the eigenmode Q includes port loading. Lengths are in µm; frequencies below are in GHz.

In [ ]:
from pathlib import Path

import gdsfactory as gf
import klayout.db as kdb
import pandas as pd
from qpdk import PDK
from qpdk.cells.chip import chip_edge
from qpdk.tech import (
    LAYER,
    LAYER_STACK,
    get_etch_section,
    material_properties,
    routing_strategies,
)
from qpdk.utils import apply_additive_metals, fill_magnetic_vortices

from gsim.common.polygon_utils import simplify_polygon
from gsim.common.stack import Layer, LayerStack
from gsim.palace import EigenmodeSim

PDK.activate()

### Settings

This large chip uses 2 µm refinement near metal and a coarser 300 µm mesh
away from it. Refinement is local; the whole chip is not meshed at 2 µm.
The narrowest conductor features are also 2 µm, so compare finer meshes
before relying on resonance frequencies or Q.

| Quantity | Count |
| --- | ---: |
| Mesh elements (tetrahedra) | 2,708,297 |
| Field DOFs, order 2 | 19,136,468 |

In [ ]:
refined_mesh_size_um = 2.0
max_mesh_size_um = 300.0
curve_tolerance_um = 0.1
chip_padding_um = 50.0
chip_edge_width_um = 100.0
port_length_um = 10.0

eigenmode_target_ghz = 5.0
num_eigenmodes = 4
mesh_fmax_ghz = 10.0

### Build the chip

Add vortex-trapping fill and leave 50 µm of ground between the layout,
including the launchers, and the 100 µm-wide etched chip-edge frame.

In [ ]:
notebook_dir = Path("nbs") if Path("nbs").is_dir() else Path.cwd()
port_layout = gf.read.from_yaml(
    notebook_dir / "data/qpdk/two_qubit_straight_bus.pic.yml",
    routing_strategies=routing_strategies,
)
chip_bounds = port_layout.dbbox().enlarged(chip_padding_um + chip_edge_width_um)
filled_layout = gf.Component()
filled_layout << fill_magnetic_vortices(
    component=port_layout,
    rectangle_size=(15.0, 15.0),
    gap=70.0,
    stagger=2,
)
edge = filled_layout << chip_edge(
    size=(chip_bounds.width(), chip_bounds.height()),
    width=chip_edge_width_um,
    layer=LAYER.M1_ETCH,
)
edge.dmove((chip_bounds.left, chip_bounds.bottom))
chip = apply_additive_metals(filled_layout)
chip.plot()

### Convert the etch mask to metal

Subtract the etch mask from the chip footprint and simplify curved boundaries.
Copy the two bus ports from the launcher pads.

In [ ]:
layout = chip.kdb_cell.layout()
etch = kdb.Region(chip.kdb_cell.begin_shapes_rec(layout.layer(*LAYER.M1_ETCH))).merged()
footprint = kdb.Region(etch.bbox())
if curve_tolerance_um > 0:
    etch = kdb.Region(
        [simplify_polygon(p, tolerance=curve_tolerance_um) for p in etch.each()]
    ).merged()
metal = footprint - etch

mesh_layers = {"M1": (201, 0), "SUBSTRATE": (202, 0), "VACUUM": (203, 0)}
mesh_component = gf.Component()
for name, region in (("M1", metal), ("SUBSTRATE", footprint), ("VACUUM", footprint)):
    mesh_component.kdb_cell.shapes(
        mesh_component.kcl.layout.layer(*mesh_layers[name])
    ).insert(region)

# Keep the exported ports on the wide launcher pads.
for port_name in ("o1", "o2"):
    mesh_component.add_port(name=port_name, port=port_layout.ports[port_name])

### Define the stack and ports

Use QPDK's silicon and vacuum thicknesses, with the PEC sheet at `z=0`.
The CPW ports span the launcher gaps and remain inside the signal pads.

In [ ]:
stack = LayerStack(pdk_name="qpdk")
for name, zmin, zmax, material, layer_type in (
    (
        "SUBSTRATE",
        -LAYER_STACK.layers["Substrate"].thickness,
        0.0,
        "silicon",
        "dielectric",
    ),
    ("M1", 0.0, 0.0, "niobium", "conductor"),
    ("VACUUM", 0.0, LAYER_STACK.layers["Vacuum"].thickness, "vacuum", "dielectric"),
):
    stack.layers[name] = Layer(
        name=name,
        gds_layer=mesh_layers[name],
        zmin=zmin,
        zmax=zmax,
        thickness=zmax - zmin,
        material=material,
        layer_type=layer_type,
    )
stack.dielectrics = [
    dict(
        name=layer.name.lower(),
        zmin=layer.zmin,
        zmax=layer.zmax,
        material=layer.material,
    )
    for layer in stack.layers.values()
    if layer.layer_type == "dielectric"
]
stack.materials = {
    "silicon": {
        "type": "dielectric",
        "permittivity": material_properties["Si"]["relative_permittivity"],
        "loss_tangent": material_properties["Si"]["loss_tangent"],
    },
    "vacuum": {"type": "dielectric", "permittivity": 1.0},
    "niobium": {"type": "conductor"},
}

In [ ]:
eigenmode_sim = EigenmodeSim(absorbing_boundary=False)
eigenmode_sim.set_geometry(mesh_component.copy())
eigenmode_sim.set_stack(stack)
eigenmode_sim.solver.linear.tolerance = 1e-9
for port_name in ("o1", "o2"):
    port = port_layout.ports[port_name]
    eigenmode_sim.add_cpw_port(
        port_name,
        layer="M1",
        s_width=float(port.dwidth),
        gap_width=get_etch_section(port.info["cross_section"]).width,
        length=port_length_um,
        offset=port_length_um / 2,
        impedance=50.0,
        excited=False,
    )

eigenmode_sim.set_eigenmode(
    num_modes=num_eigenmodes,
    target=eigenmode_target_ghz * 1e9,
    save=num_eigenmodes,
)

### Generate the eigenmode mesh

Generate the mesh using 2 µm refinement. Embed both port surfaces and inspect
`P1` and `P2` before submitting the eigenmode job.

The simulation API requires a mesh output directory.
Cloud downloads use gsim's default `sim-data-<job-name>` folders.
The model uses `absorbing_boundary=False` for reflecting PMC outer boundaries.

In [ ]:
eigenmode_sim.set_output_dir("palace-qpdk-large-chip-eigenmode")
eigenmode_mesh = eigenmode_sim.mesh(
    refined_mesh_size=refined_mesh_size_um,
    max_mesh_size=max_mesh_size_um,
    fmax=mesh_fmax_ghz * 1e9,
    planar_conductors=True,
)
print(f"Tetrahedral elements: {eigenmode_mesh.mesh_stats['tetrahedra']:,}")

In [ ]:
eigenmode_sim.plot_mesh(
    show_groups=["M1", "P1", "P2"],
    interactive=False,
)

### Run the four-mode eigenmode simulation

Run the eigenmode object on GDSFactory+ cloud. Its `.run()` method prepares
the configuration, reuses a completed job when the inputs match, and downloads
the results. `save=num_eigenmodes` enables ParaView field export for all
requested modes; the fields are included in the downloaded results.

In [ ]:
eigenmode_results = eigenmode_sim.run(check_cache=True)

In [ ]:
eigenmodes = pd.read_csv(eigenmode_results["eig.csv"], skipinitialspace=True)
eigenmodes.columns = eigenmodes.columns.str.strip()
print(
    f"Converged eigenmodes: {len(eigenmodes)} / "
    f"{eigenmode_sim.solver.eigenmode.num_modes} requested"
)
eigenmodes